# Part 2 — Relating racialised discourse to socioeconomic/political data

**Scope**: descriptive, correlational, and predictive — Spearman correlations with multiple-comparison correction, non-parametric group comparisons, proper spatial autocorrelation, and (Part 3, §3.1 onward) a negative-binomial regression model, run once the IAA/robustness work established how much to trust the `racialised` label itself.

## Leads from Part 1 (`descriptive_analysis.ipynb`) carried into this pass

Part 1's closing "Open threads" section flagged several leads that need socioeconomic/political data to resolve. This pass picks up one directly:

- **Arab ethnicity-share** (§2.4g below): the only `ethnicity_group4` share that survives controlling for `population_density_per_km2` (rho=+0.132 vs. raw +0.211) — `Germany`'s link disappears entirely under that control and `Türkiye`'s weakens to non-significance. That check only ever controlled for density; it has not yet been run against this notebook's own deprivation/political predictors.

**Deliberately not pursued in this pass** (so scope stays bounded):
- The `Staff`-theme rural-concentration puzzle — Part 1 resolved it as a genuine within-group effect, not a compositional one, but *why* places differ remains an open, qualitative lead, not something a socioeconomic correlate is likely to explain.
- `approach` has near-chance inter-annotator agreement (κ≈0-0.09) in Part 1 — carried forward here only as a caution, not re-tested, wherever `approach` would otherwise appear (it doesn't feature in this notebook's own crosstabs, so this is a non-issue in practice, noted for completeness).


# Part 0 — Data validation

Before treating any socioeconomic/political predictor as informative for §2.4 onward's correlations, this Part checks two things end-to-end: (1) that the raw source data survived cleaning without corruption, and (2) that the derivations this notebook performs on top of that cleaned data (raw total/rate → per-capita or percentage predictor) actually preserve real information rather than just repackaging it. This was prompted directly by a supervisor note: confirm the data isn't corrupt, and confirm that what we do with the substantive predictor columns is actually making a difference. In scope: raw columns 7–48 of `federal_baseline_eng.csv` -- everything from `Population – Germans (in 1,000s)` through `Unemployment rate November 2024 – men` -- i.e. the whole predictor-ingredient column set, excluding only the identifying columns (1–3), municipalities/area (4–5), total population itself (6), and the unemployment-breakdown tail (49–51).

**What's below, in order:** §0.1 checks the raw-to-cleaned pipeline directly (decimal-comma parsing, `(in 1,000s)` scaling, range sanity, unexpected nulls), using `federal_baseline_eng.csv`/`federal_baseline_eng_clean.csv` before any of this notebook's own derivations touch the data. §2.1–§2.3b then build this notebook's own analysis table and predictor set -- kept under their original numbering since later sections (§2.4 onward) cross-reference them by that number -- and include an existing denominator/merge-integrity audit (§2.3b), promoted here to sit before any correlation work rather than mid-notebook where it was added earlier. §0.2 closes Part 0 by checking whether this notebook's rate→count→rate/per-capita derivations actually add information over the originally-published rates, folding in §2.3b's Berlin/Hamburg finding as the sharpest case where they don't. Everything from `# Correlations, relationships, and spatial analysis` onward assumes Part 0 has already passed.

## 0. Setup — continuing from Part 1

This notebook continues directly from `descriptive_analysis.ipynb`, which built and verified the place→constituency spatial join and the constituency-level aggregate table. Rather than repeating that join, we load its two saved outputs (`const_agg.parquet`, `racialised_reviews_geo.parquet`) plus the socioeconomic/political CSV and the constituency shapefile (needed again here only for its polygon geometry, to draw maps and compute spatial autocorrelation — not to redo the point-in-polygon join itself).

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)

ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
SOCIO_DIR = ANALYSIS_DIR / "socio-economic_data"
OUTPUTS_DIR = ANALYSIS_DIR / "notebook_outputs"
FIGURES_DIR = ANALYSIS_DIR / "figures"
PROJECT_ROOT = Path("/home/ethel/Desktop/swimming_pool_project")
SHAPEFILE_PATH = PROJECT_ROOT / "btw25_geometrie_wahlkreise_shp" / "btw25_geometrie_wahlkreise_shp.shp"

socio_political = pd.read_csv(SOCIO_DIR / "socio_political_data.csv")
const_agg = pd.read_parquet(OUTPUTS_DIR / "const_agg.parquet")
racialised_df = pd.read_parquet(OUTPUTS_DIR / "racialised_reviews_geo.parquet")

print(f"political/socioeconomic: {socio_political.shape[0]} constituencies x {socio_political.shape[1]} cols")
print(f"const_agg (from Part 1): {const_agg.shape[0]} constituencies x {const_agg.shape[1]} cols")
print(f"racialised_df (from Part 1): {racialised_df.shape[0]} reviews x {racialised_df.shape[1]} cols")

assert const_agg.shape[0] == 299, "expected 299 constituencies"
assert racialised_df.shape[0] == 7_768, "racialised review count drifted from Part 1"


## 2.1 Building the constituency-level analysis table

`const_agg` (built in Part 1) already carries `constituency_code`/`constituency`/`state` in the same form as `federal_baseline_eng_selected_political.csv` (both ultimately keyed off the same `federal_baseline_eng_selected_political.csv` backbone, so there's no naming mismatch left to fix here — that mismatch was between the raw shapefile fields (`WKR_NR`/`LAND_NR`) and this file, and we already resolved it in Section 1). We merge in every other socioeconomic/political column.


In [ ]:
const_agg.rename(columns={'constituency_nr': 'constituency_code', 'constituency_name': 'constituency'}, inplace=True)

const_agg

### 0. Pinning down what the racialsied rate actually is

We already set a threshold for the racialised rate: constituencies with 10+ raciliased reviews and 100+ social reviews and we had 247 constituencies

In [ ]:
const_agg_r = const_agg[(const_agg['social_reviews'] >= 100) & (const_agg['racialised_reviews'] >=10)]
const_agg_r.shape


In [ ]:
# how skewed is the distribution of racialised rate across constituencies? side by side with the filtered version
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=const_agg, x="racialised_rate", bins=30, color="skyblue", edgecolor="black", ax=axes[0])
axes[0].set_title("Distribution of Racialised Rate Across Constituencies (All)", fontsize=12)
axes[0].set_xlabel("Racialised Rate", fontsize=12)
axes[0].set_ylabel("Count of Constituencies", fontsize=12)  


sns.histplot(data=const_agg_r, x="racialised_rate", bins=30, color="salmon", edgecolor="black", ax=axes[1])
axes[1].set_title("Distribution of Racialised Rate Across Constituencies (Filtered)", fontsize=12)
axes[1].set_xlabel("Racialised Rate", fontsize=12)
axes[1].set_ylabel("Count of Constituencies", fontsize=12)
    

plt.show()

In [ ]:
const_agg["social_reviews"].describe()

In [ ]:
#merge the socio-political data with the aggregated constituency data

analysis_df = const_agg.merge(socio_political,on=["state", "constituency_code", "constituency"], how="left")
analysis_df.shape

### Proceeding with all constituencies

What do the socio economic and political variables look like?

In [ ]:
#selecting possible predictors
#merging some age groups to get a more general picture of the older population
analysis_df['age_60_plus_pct'] = analysis_df['age_60_74_pct'] + analysis_df['age_75_plus_pct']
analysis_df['second_vote_left_bsw'] = analysis_df['second_vote_left_pct'] + analysis_df['second_vote_bsw_pct']
analysis_df['log_population_density'] = np.log(analysis_df['population_density_per_km2'])
analysis_df['disposable_income_1k'] = analysis_df['disposable_income_eur_per_inhabitant'] / 1000
east_states = ['Brandenburg', 'Mecklenburg-Vorpommern',
               'Sachsen', 'Sachsen-Anhalt', 'Thüringen']
analysis_df['is_east'] = analysis_df['state'].isin(east_states).astype(int)

socio_predictors = ['foreign_population_pct', #migrationa and foreigner presence
                     'log_population_density',  #size and settelment 
                     'unemployment_total_pct', #dependency on social welfare
                     'sgb_ii_recipients_per_1k', #dependency on social welfare  # drop after correlation check
                     'school_grads_higher_education_entrance_pct', #education
                     'age_60_plus_pct', #age stucture
                     'disposable_income_1k' #affluence
                     ]
political_predictors = ['second_vote_afd_pct', # right-wing populist party
                        'second_vote_left_bsw', # left-wing parties
                        'second_vote_greens_pct', # green party
                        'turnout_pct'
                        ]

structural = ['is_east']
predictors = socio_predictors + political_predictors

In [ ]:
def repeated_value_clusters(df, column):
    """Within each state, find groups of >=2 constituencies sharing an
    identical value for `column` -- same logic as the corrected
    audit_state_repeated_columns.py, applied constituency-by-constituency
    (a shared value usually means a district/Kreis-level stat got duplicated
    across every constituency that splits that district, not a real
    constituency-level difference)."""
    clusters = []
    for state, sub in df.groupby('state'):
        for value, g in sub.groupby(column):
            if len(g) > 1:
                clusters.append({'state': state, 'value': value, 'constituencies': g['constituency'].tolist()})
    return clusters


for col in socio_predictors:
    clusters = repeated_value_clusters(analysis_df, col)
    n_affected = sum(len(c['constituencies']) for c in clusters)
    print(f"{col}: {n_affected} constituencies affected across {len(clusters)} cluster(s)")
    for c in sorted(clusters, key=lambda c: -len(c['constituencies'])):
        print(f"    {c['state']} [{len(c['constituencies'])}]: {', '.join(c['constituencies'])}")
    print()


- `unemployment_total_pct` — repeated in Berlin (12/12 constituencies) and Hamburg (6/6)
- `sgb_ii_recipients_per_1k` — repeated in Berlin (12/12) and Hamburg (6/6)
- `age_60_plus_pct (= age_60_74_pct + age_75_plus_pct)` — both components repeated in Berlin (12/12) and Hamburg (6/6)
- `disposable_income_1k (from disposable_income_eur_per_inhabitant)` — repeated in Berlin (12/12) and Hamburg (6/6)
- `school_grads_higher_education_entrance_pct — repeated in Hamburg only (6/6)`; Berlin actually varies (9 distinct values across 12)
Genuinely vary per constituency everywhere (no issue): foreign_population_pct, log_population_density, all four political_predictors (second_vote_afd_pct, second_vote_left_bsw, second_vote_greens_pct, turnout_pct).

Bremen: zero repetition across any variable — its 2 constituencies (Bremen city / Bremerhaven) are genuinely distinct throughout.

In [ ]:
#plot histograms of the predictors to see their distributions
cols = socio_predictors + political_predictors + structural # 12 items
fig, axes = plt.subplots(6,2, figsize=(12, 20), constrained_layout=True)

for col, ax in zip(cols, axes.flat):
    sns.histplot(data=analysis_df, x=col, bins=30,
                 color="lightblue", edgecolor="black", ax=ax)
    ax.set_title(f"Dist. {col}", fontsize=10)
    ax.set_xlabel("")
    ax.set_xlabel("")

fig.supylabel("Count of Constituencies")    

In [ ]:
# checking the thresholds for the socio-political predictors to see which constituencies exceed them
checks = {
    'foreign_population_pct': 28,
    'disposable_income_1k': 30,
    'school_grads_higher_education_entrance_pct': 50,
    'log_population_density': 8.5,
    'sgb_ii_recipients_per_1k': 150,
}
for col, thresh in checks.items():
    hits = analysis_df[analysis_df[col] > thresh]
    print(f"\n{col} > {thresh}  ({len(hits)} constituencies)")
    print(hits[['constituency', 'state', col, 'social_reviews', 'racialised_rate']]
          .sort_values(col, ascending=False).to_string(index=False))

## Correlation among predicators

In [ ]:
def pair_table(df, cols, label):
    c = df[cols].corr(method='spearman')
    m = np.triu(np.ones(c.shape), k=1).astype(bool)
    s = c.where(m).stack()
    s.index = [f"{a}  ×  {b}" for a, b in s.index]
    return s.rename(label)

cols = socio_predictors + political_predictors
tbl = pd.concat([
    pair_table(analysis_df, cols, 'all'),
    pair_table(analysis_df[analysis_df.is_east == 0], cols, 'west'),
    pair_table(analysis_df[analysis_df.is_east == 1], cols, 'east'),
], axis=1).round(2)

tbl['verdict'] = np.select(
    [tbl[['all','west','east']].abs().min(axis=1) > 0.7,
     (tbl['all'].abs() > 0.6) & (tbl['west'].abs() < 0.4),
     tbl['all'].abs() > 0.6],
    ['REDUNDANT — drop one', 'east/west artefact — keep both', 'watch'],
    default='fine')

print(tbl[tbl.verdict != 'fine'].sort_values('all', key=abs, ascending=False).to_string())

## Checking duplication and repeated values across in the socio economic data

In [ ]:
check_cols = ['unemployment_total_pct', 'sgb_ii_recipients_per_1k',
              'school_grads_higher_education_entrance_pct',
              'age_60_74_pct', 'age_75_plus_pct', 'disposable_income_1k']

dup_groups = analysis_df.groupby(check_cols)['constituency'].apply(list)
dup_groups = dup_groups[dup_groups.apply(len) > 1]

for vals, consts in dup_groups.items():
    state = analysis_df.loc[analysis_df['constituency'] == consts[0], 'state'].iloc[0]
    print(f"{state}: {consts}")

In [ ]:
n_affected = dup_groups.apply(len).sum()
print(f"{n_affected} constituencies in {len(dup_groups)} clusters\n")

In [ ]:
# state per cluster, to see if the duplicates are within the same state or across states
dup_groups = analysis_df.groupby(check_cols + ['state'])['constituency'].apply(list)
dup_groups = dup_groups[dup_groups.apply(len) > 1]

n_affected = dup_groups.apply(len).sum()
print(f"{n_affected} constituencies in {len(dup_groups)} clusters\n")

for vals, consts in dup_groups.sort_values(key=lambda s: s.apply(len), ascending=False).items():
    print(f"{vals[-1]:24s} [{len(consts)}]  {consts}")

In [ ]:
#flag constituencies that are part of a cluster of duplicates, to be able to filter them out in the analysis
flagged = {c for consts in dup_groups for c in consts}
analysis_df['in_split_city'] = analysis_df['constituency'].isin(flagged).astype(int)
print(analysis_df['in_split_city'].sum(), "flagged")

In [ ]:
print(analysis_df[analysis_df.state == 'Berlin']
      [['constituency'] + check_cols].to_string(index=False))

## Dropping Redundant  Predictors

In [ ]:
socio_predictors = [
    'foreign_population_pct',
    'log_population_density',
    'sgb_ii_recipients_per_1k',
    'school_grads_higher_education_entrance_pct',
    'age_60_plus_pct',
    'disposable_income_1k'
]
political_predictors = [
    'second_vote_afd_pct',
    'second_vote_left_bsw',
    'turnout_pct'
]
structural = ['is_east', 'in_split_city']

predictors = socio_predictors + political_predictors

## Predicators against the outcome

In [ ]:
from scipy.stats import spearmanr

def summarise(df, cols, label):
    rows = {}
    for c in cols:
        r, p = spearmanr(df[c], df['racialised_rate'])
        rows[c] = pd.Series({f'{label}_rho': round(r, 2),
                             f'{label}_p':   p})
    return pd.DataFrame(rows).T

def strength(r, n):
    cutoff = 2 / np.sqrt(n)          # roughly "distinguishable from zero"
    if abs(r) < cutoff:  return 'none'
    if abs(r) < 0.30:    return 'weak'
    if abs(r) < 0.50:    return 'moderate'
    return 'strong'

cand = ['foreign_population_pct', 'log_population_density',
        'sgb_ii_recipients_per_1k', 'turnout_pct',
        'school_grads_higher_education_entrance_pct',
        'age_60_plus_pct', 'second_vote_afd_pct',
        'disposable_income_1k', 'second_vote_left_bsw']


west = analysis_df[analysis_df.is_east == 0]
east = analysis_df[analysis_df.is_east == 1]

tbl = pd.concat([summarise(analysis_df, cand, 'all'),
                 summarise(west, cand, 'west'),
                 summarise(east, cand, 'east')], axis=1)

tbl['west_reads'] = [f"{strength(r, 251)} {'up' if r > 0 else 'down'}"
                     for r in tbl.west_rho]
tbl['east_reads'] = [f"{strength(r, 48)} {'up' if r > 0 else 'down'}"
                     for r in tbl.east_rho]
tbl['agrees'] = np.where(np.sign(tbl.west_rho) == np.sign(tbl.east_rho),
                         'same direction', 'OPPOSITE')

print(tbl[['west_rho', 'west_reads', 'east_rho', 'east_reads', 'agrees']]
      .sort_values('west_rho', key=abs, ascending=False)
      .to_string())

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 14), constrained_layout=True)

for col, ax in zip(predictors, axes.flat):
    sns.scatterplot(data=analysis_df, x=col, y='racialised_rate',
                    hue='is_east', palette={0: 'steelblue', 1: 'firebrick'},
                    size='social_reviews', sizes=(15, 150),
                    alpha=.55, legend=False, ax=ax)
    sns.regplot(data=analysis_df, x=col, y='racialised_rate',
                lowess=True, scatter=False,
                line_kws={'color': 'black', 'lw': 2}, ax=ax)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel(''); ax.set_ylabel('')

fig.supylabel('racialised rate')
fig.suptitle('Predictors vs racialised rate  (red = East, size = social reviews)')

In [ ]:
west = analysis_df[analysis_df['is_east'] == 0]

fig, axes = plt.subplots(3, 3, figsize=(16, 14), constrained_layout=True)
for col, ax in zip(predictors, axes.flat):
    sns.regplot(data=west, x=col, y='racialised_rate', lowess=True,
                scatter_kws={'alpha': .4, 's': 20},
                line_kws={'color': 'black'}, ax=ax)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel(''); ax.set_ylabel('')

fig.suptitle('West only (n≈251)')
fig.supylabel('racialised rate')

In [ ]:
east = analysis_df[analysis_df['is_east'] == 1]

fig, axes = plt.subplots(3, 3, figsize=(16, 14), constrained_layout=True)
for col, ax in zip(predictors, axes.flat):
    sns.regplot(data=east, x=col, y='racialised_rate', lowess=True,
                scatter_kws={'alpha': .4, 's': 20},
                line_kws={'color': 'black'}, ax=ax)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel(''); ax.set_ylabel('')
fig.suptitle('East only (n≈48)')
fig.supylabel('racialised rate')

### Step 5 — Predictor vs. racialised rate: shape reading

Read from LOWESS-smoothed scatter plots (all 299; West n≈251; East n≈48).
East shapes rest on 48 constituencies and are indicative only.

| Predictor | West shape | East shape | Consistent? | Decision |
|---|---|---|---|---|
| `log_population_density` | Rising, near-linear, tight spread | Rising, near-linear | Yes | **Keep** — best-behaved predictor |
| `foreign_population_pct` | Rising, flattens above ~25% | Rising | Yes | **Keep** |
| `sgb_ii_recipients_per_1k` | Rising, strongly convex above ~100/1k | Flat to ~70, slight rise after | No | **Keep + East interaction** |
| `turnout_pct` | Falling steeply below 78%, flat above 82% | Flat to slightly rising | No | **Keep + East interaction** |
| `school_grads_higher_education_entrance_pct` | Near-flat, slight rise | Near-flat, slight rise | Yes | Keep as control — weak |
| `age_60_plus_pct` | Slight fall | Falls to ~36%, then flattens | Roughly | Keep as control — weak |
| `second_vote_afd_pct` | Flat to slightly negative | Clearly negative | Yes (both ≤ 0) | Keep — expect null or negative, not the anticipated positive |
| `disposable_income_1k` | L-shaped; slope driven by ~4 constituencies below 21k, flat above 22k | U-shaped; range only 20.4–23.8k | No | **Drop** — no usable linear shape; duplicates deprivation |
| `second_vote_left_bsw` | Rising, convex — mirrors `sgb_ii` | Flat | No | **Drop** — ρ = 0.84 with `sgb_ii` in West |
| `unemployment_total_pct` | Rising, convex | Flat / U-shaped | No | **Dropped at Step 4** — ρ = 0.96 with `sgb_ii` |
| `second_vote_greens_pct` | — | — | — | **Dropped at Step 4** — ρ = −0.89 with AfD |

**General observations**

- Several relationships are convex on the raw rate scale (`sgb_ii`, `left_bsw`). This is consistent with multiplicative rather than additive effects and is handled automatically by a count model with a log link — supports the Step 7 model choice.
- East and West occupy different ranges on `sgb_ii`, `age_60_plus_pct` and `turnout_pct` with minimal overlap. The pooled smoother partly traces the East–West gap rather than a within-group relationship. An intercept shift (`is_east`) is insufficient; slope interactions are required.
- Large markers (high social-review counts) sit close to the smoothers across all panels; wide scatter is concentrated among small markers. Relationships are not artefacts of imprecisely measured constituencies.

**Carried into Step 7**

```
predictors   = foreign_population_pct, log_population_density,
               sgb_ii_recipients_per_1k, turnout_pct,
               school_grads_higher_education_entrance_pct,
               age_60_plus_pct, second_vote_afd_pct
structural   = is_east, in_split_city
interactions = is_east × sgb_ii_recipients_per_1k
               is_east × turnout_pct
```

In [ ]:
from scipy.stats import spearmanr

def rate_corr(df, cols, label):
    out = {}
    for c in cols:
        r, p = spearmanr(df[c], df['racialised_rate'])
        out[c] = round(r, 2)
    return pd.Series(out, name=label)

cand = ['foreign_population_pct', 'log_population_density',
        'sgb_ii_recipients_per_1k', 'turnout_pct',
        'school_grads_higher_education_entrance_pct',
        'age_60_plus_pct', 'second_vote_afd_pct',
        'disposable_income_1k', 'second_vote_left_bsw']

summary = pd.concat([
    rate_corr(analysis_df, cand, 'all_299'),
    rate_corr(analysis_df[analysis_df.is_east == 0], cand, 'west_251'),
    rate_corr(analysis_df[analysis_df.is_east == 1], cand, 'east_48'),
], axis=1)

summary['sign_flips'] = np.sign(summary.west_251) != np.sign(summary.east_48)
print(summary.sort_values('west_251', key=abs, ascending=False).to_string())

## Structure not Captured by predictors
The predictors all vary continuously. But constituencies also sit inside states, and states differ for reasons no predictor captures — different pool infrastructure, different Google Maps usage, different regional cultures of complaint. If most of the variation in your outcome sits between states rather than within them, a model that ignores this will report confidence intervals that are too narrow.

In [ ]:
#--- a. rate by rate
order = (analysis_df.groupby('state')['racialised_rate']
         .median().sort_values().index)

fig, ax = plt.subplots(figsize=(13, 6), constrained_layout=True)
sns.boxplot(data=analysis_df, x='state', y='racialised_rate',
            order=order, color='lightsteelblue', fliersize=0, ax=ax)
sns.stripplot(data=analysis_df, x='state', y='racialised_rate',
              order=order, hue='is_east',
              palette={0: 'steelblue', 1: 'firebrick'},
              size=4, alpha=.7, legend=False, ax=ax)
ax.axhline(analysis_df.racialised_rate.mean(), ls='--', c='k', lw=1)
ax.tick_params(axis='x', rotation=60)

In [ ]:
print(analysis_df.groupby('state')
      .agg(n=('racialised_rate', 'size'),
           median=('racialised_rate', 'median'),
           iqr=('racialised_rate', lambda s: s.quantile(.75) - s.quantile(.25)))
      .sort_values('median').round(4).to_string())

In [ ]:
##b — How much variation is between states
overall_var = analysis_df['racialised_rate'].var()
within_var  = analysis_df.groupby('state')['racialised_rate'].var().mean()
print(f"share of variance between states: {1 - within_var/overall_var:.0%}")

In [ ]:
#---c — East versus West directly
print(analysis_df.groupby('is_east')['racialised_rate']
      .describe()[['count', 'mean', '50%', 'std']].round(4))

In [ ]:
#--d — Spatial clustering
import geopandas as gpd

gdf = gpd.read_file(SHAPEFILE_PATH)
gdf = gdf.merge(analysis_df, left_on='WKR_NR', right_on='constituency_code')

fig, ax = plt.subplots(figsize=(9, 11), constrained_layout=True)
gdf.plot(column='racialised_rate', cmap='YlOrRd', legend=True,
         edgecolor='white', linewidth=.3, ax=ax)
ax.set_axis_off()
ax.set_title('Racialised rate by constituency')

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran

w = Queen.from_dataframe(gdf)
w.transform = 'r'
mi = Moran(gdf['racialised_rate'].values, w)
print(f"Moran's I = {mi.I:.3f}   p = {mi.p_sim:.4f}")


In [ ]:
resid = gdf['racialised_rate'] - gdf['racialised_rate'].mean()   # placeholder
resid

In [ ]:
analysis_df.info()

## Choosing a model

## Fit the model
Why negative binomial

Model the racialised count, with social reviews as the base. Two reasons, both established earlier: it weights each constituency by how well-measured it is (307 vs 11,097 reviews), and its log link assumes multiplicative effects, which straightens the convex shapes you saw in the scatter plots.

In [ ]:

#--setup
import statsmodels.api as sm
import statsmodels.formula.api as smf


df = analysis_df.copy()
df = df.rename(columns={'age_60+_pct': 'age_60_plus_pct'})
df['offset'] = np.log(df['social_reviews'])

for c in predictors:
    df[c + '_c'] = df[c] - df[c].mean()

formula = ("racialised_reviews ~ "
           "foreign_population_pct_c + log_population_density_c "
           "+ sgb_ii_recipients_per_1k_c + turnout_pct_c "
           "+ school_grads_higher_education_entrance_pct_c "
           "+ age_60_plus_pct_c + second_vote_afd_pct_c "
           "+ is_east + in_split_city "
           "+ is_east:sgb_ii_recipients_per_1k_c "
           "+ is_east:turnout_pct_c")

In [ ]:
##-- fit poisson  and check the dispersion
p = smf.glm(formula, data=df, family=sm.families.Poisson(),
            offset=df['offset']).fit()
disp = p.pearson_chi2 / p.df_resid
print("dispersion:", round(disp, 2))

In [ ]:
##-- estimate alpha for negative binomial
mu = p.mu
aux = sm.OLS(((df['racialised_reviews'] - mu)**2 - mu) / mu, mu).fit()
alpha_hat = float(aux.params.iloc[0])
print("alpha:", round(alpha_hat, 4))

m = smf.glm(formula, data=df,
            family=sm.families.NegativeBinomial(alpha=alpha_hat),
            offset=df['offset']).fit(cov_type='HC1')

In [ ]:
# 7c, after alpha_hat is computed
fam = sm.families.NegativeBinomial(alpha=alpha_hat)

In [ ]:
##-- — Compare standard error choices
m_hc  = smf.glm(formula, data=df, family=fam, offset=df['offset']).fit(cov_type='HC1')
m_cl  = smf.glm(formula, data=df, family=fam, offset=df['offset']).fit(
            cov_type='cluster', cov_kwds={'groups': df['state']})

comp = pd.DataFrame({'coef': m_hc.params.round(4),
                     'se_hc1': m_hc.bse.round(4),
                     'se_cluster': m_cl.bse.round(4)})
print(comp.to_string())

In [ ]:
##.-- — Read the results
res = pd.DataFrame({'rate_ratio': np.exp(m.params).round(3),
                    'ci_low':  np.exp(m.conf_int()[0]).round(3),
                    'ci_high': np.exp(m.conf_int()[1]).round(3),
                    'p': m.pvalues.round(4)})
print(res.to_string())

In [ ]:
##--- simplify: Both interactions are non-significant. Refit without them, and without the four dead predictors, then compare
formula2 = ("racialised_reviews ~ foreign_population_pct_c "
            "+ log_population_density_c + sgb_ii_recipients_per_1k_c "
            "+ is_east + in_split_city")

m2 = smf.glm(formula2, data=df, family=fam, offset=df['offset']).fit(
        cov_type='cluster', cov_kwds={'groups': df['state']})
print(pd.DataFrame({'rate_ratio': np.exp(m2.params).round(3),
                    'p': m2.pvalues.round(4)}).to_string())
print("AIC full:", m.aic.round(1), " reduced:", m2.aic.round(1))

In [ ]:
# does foreign_population_pct flatten at the top, as the plot suggested?
df['foreign_sq'] = df['foreign_population_pct_c']**2
m3 = smf.glm(formula2 + " + foreign_sq", data=df, family=fam,
             offset=df['offset']).fit(cov_type='cluster',
                                      cov_kwds={'groups': df['state']})
print(m3.aic.round(1), m3.pvalues['foreign_sq'].round(4))

In [ ]:
formula_spline = ("racialised_reviews ~ bs(foreign_population_pct_c, df=4) "
                  "+ log_population_density_c + sgb_ii_recipients_per_1k_c "
                  "+ is_east + in_split_city")

In [ ]:
from patsy import dmatrix   # bs() comes from patsy, available in the formula

formula_spline = ("racialised_reviews ~ bs(foreign_population_pct_c, df=4) "
                  "+ log_population_density_c + sgb_ii_recipients_per_1k_c "
                  "+ is_east + in_split_city")

m_spline = smf.glm(formula_spline, data=df, family=fam,
                   offset=df['offset']).fit(cov_type='cluster',
                                            cov_kwds={'groups': df['state']})

print("AIC straight line:", round(m2.aic, 1))
print("AIC spline:       ", round(m_spline.aic, 1))

In [ ]:
df['foreign_sq'] = df['foreign_population_pct_c']**2

m_sq = smf.glm(formula2 + " + foreign_sq", data=df, family=fam,
               offset=df['offset']).fit(cov_type='cluster',
                                        cov_kwds={'groups': df['state']})

print("AIC:", round(m_sq.aic, 1), " p on squared term:",
      round(m_sq.pvalues['foreign_sq'], 4))

In [ ]:
def partial_plot(model, df, var, ax, xlabel=None):
    raw = var.replace('_c', '')
    mean = df[raw].mean()

    grid_raw = np.linspace(df[raw].min(), df[raw].max(), 100)
    grid_c   = grid_raw - mean

    newd = pd.DataFrame({c: [df[c].mean()] * 100 for c in
                         ['foreign_population_pct_c', 'log_population_density_c',
                          'sgb_ii_recipients_per_1k_c', 'is_east', 'in_split_city']})
    newd[var] = grid_c
    newd['offset'] = np.log(1000)

    pred = model.get_prediction(newd, offset=newd['offset']).summary_frame()

    # actual data behind the line
    ax.scatter(df[raw], df['racialised_rate'], s=8, alpha=.25, color='steelblue')
    ax.plot(grid_raw, pred['mean'] / 1000, 'k-', lw=2)
    ax.fill_between(grid_raw, pred['mean_ci_lower']/1000,
                    pred['mean_ci_upper']/1000, alpha=.25, color='steelblue')
    ax.set_xlabel(xlabel or raw)
    ax.set_ylabel('predicted racialised rate')
    ax.set_ylim(0, .06)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
labels = ['foreign population (%)',
          'population density (log per km²)',
          'benefit recipients per 1,000']
for v, lab, ax in zip(['foreign_population_pct_c', 'log_population_density_c',
                       'sgb_ii_recipients_per_1k_c'], labels, axes):
    partial_plot(m2, df, v, ax, lab)

In [ ]:
print(m_sq.params['foreign_sq'], np.exp(m_sq.params['foreign_sq']))

## Check for joint redundancy
At Step 2 you found `foreign_population_pct` and `log_population_density` correlate at 0.78. That was a warning. Now that the model is fitted, you can check whether it actually caused a problem — and pairwise correlation can miss cases where three predictors are jointly redundant while no pair looks bad

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrix

X = dmatrix("foreign_population_pct_c + log_population_density_c "
            "+ sgb_ii_recipients_per_1k_c + is_east + in_split_city",
            data=df, return_type='dataframe')

vif = pd.DataFrame({
    'variable': X.columns,
    'vif': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
}).round(2)

print(vif[vif.variable != 'Intercept'].to_string(index=False))

In [ ]:
base = ['foreign_population_pct_c', 'log_population_density_c',
        'sgb_ii_recipients_per_1k_c', 'is_east', 'in_split_city']

rows = {}
for drop in ['foreign_population_pct_c', 'log_population_density_c',
             'sgb_ii_recipients_per_1k_c']:
    f = "racialised_reviews ~ " + " + ".join(c for c in base if c != drop)
    mm = smf.glm(f, data=df, family=fam, offset=df['offset']).fit(
             cov_type='cluster', cov_kwds={'groups': df['state']})
    rows[f'without_{drop}'] = np.exp(mm.params).round(3)

rows['full'] = np.exp(m2.params).round(3)
print(pd.DataFrame(rows).to_string())

## Check the fit
The model has produced numbers. This step asks whether they're trustworthy: whether the fit is systematically wrong anywhere, whether the spatial clustering survived, and whether a handful of constituencies are driving the result.

In [ ]:
##-- residuals vs fitted
df['fitted'] = m2.fittedvalues
df['resid_pearson'] = m2.resid_pearson

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

axes[0].scatter(df.fitted, df.resid_pearson, s=15, alpha=.5)
axes[0].axhline(0, c='k', lw=1)
axes[0].set_xlabel('fitted count'); axes[0].set_ylabel('Pearson residual')

axes[1].scatter(df.social_reviews, df.resid_pearson, s=15, alpha=.5)
axes[1].axhline(0, c='k', lw=1)
axes[1].set_xscale('log')
axes[1].set_xlabel('social reviews (log)'); axes[1].set_ylabel('Pearson residual')

In [ ]:
##--- residual by state
order = df.groupby('state')['resid_pearson'].median().sort_values().index

fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
sns.boxplot(data=df, x='state', y='resid_pearson', order=order,
            color='lightsteelblue', ax=ax)
ax.axhline(0, c='k', lw=1)
ax.tick_params(axis='x', rotation=60)

In [ ]:
#moran's I on residuals
gdf2 = gdf.merge(df[['constituency_code', 'resid_pearson']],
                 on='constituency_code')

w = Queen.from_dataframe(gdf2)
w.transform = 'r'
mi_resid = Moran(gdf2['resid_pearson'].values, w)
print(f"Moran's I on residuals = {mi_resid.I:.3f}   p = {mi_resid.p_sim:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
gdf2.plot(column='resid_pearson', cmap='RdBu_r', legend=True,
          vmin=-3, vmax=3, edgecolor='white', linewidth=.3, ax=ax)
ax.set_axis_off(); ax.set_title('Model residuals')

In [ ]:
##--- influential constituencies
inf = pd.DataFrame({
    'constituency': df.constituency,
    'state': df.state,
    'rate': df.racialised_rate.round(4),
    'resid': df.resid_pearson.round(2),
    'cooks': m2.get_influence().cooks_distance[0].round(3)
}).sort_values('cooks', ascending=False)

print(inf.head(12).to_string(index=False))

In [ ]:
#drop = ['Duisburg II', 'Gelsenkirchen']
drop = ['Offenburg', 'Lörrach – Müllheim']
m_nodrop = smf.glm(formula2, data=df[~df.constituency.isin(drop)],
                   family=fam, offset=np.log(df.loc[~df.constituency.isin(drop),
                                                    'social_reviews'])
                   ).fit(cov_type='cluster',
                         cov_kwds={'groups': df.loc[~df.constituency.isin(drop), 'state']})

print(pd.DataFrame({'full': np.exp(m2.params).round(3),
                    'without_extremes': np.exp(m_nodrop.params).round(3)}).to_string())

In [ ]:
print(df[df.constituency.isin(['Offenburg', 'Lörrach – Müllheim'])]
      [['constituency', 'social_reviews', 'racialised_reviews', 'racialised_rate',
        'foreign_population_pct', 'sgb_ii_recipients_per_1k',
        'population_density_per_km2']].to_string(index=False))

## Robustness

In [ ]:
def refit(subset, label, drop_terms=()):
    d = df[subset]
    terms = ['foreign_population_pct_c', 'log_population_density_c',
             'sgb_ii_recipients_per_1k_c', 'is_east', 'in_split_city']
    terms = [t for t in terms if t not in drop_terms]
    f = "racialised_reviews ~ " + " + ".join(terms)

    mm = smf.glm(f, data=d, family=fam, offset=np.log(d['social_reviews'])
                 ).fit(cov_type='cluster', cov_kwds={'groups': d['state']})
    return np.exp(mm.params).round(3).rename(f"{label}_{len(d)}")

checks = pd.concat([
    np.exp(m2.params).round(3).rename('main_299'),
    refit(df.in_split_city == 0,              'no_split',  ['in_split_city']),
    refit(df.racialised_reviews >= 10,        'ten_plus'),
    refit(df.is_east == 0,                    'west_only', ['is_east']),
    refit(df.state != 'Nordrhein-Westfalen',  'no_nrw'),
], axis=1)

print(checks.to_string())